> ### **Targets Clean**

In [0]:
bronze_df = spark.table("accenture_final_project.bronze_layer.targets")
display(bronze_df)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, to_date, initcap, trim
from pyspark.sql.types import IntegerType, StringType

In [0]:
import pyspark.sql.functions as F

# 1.Read from Bronze layer
bronze_df_targets = spark.table("accenture_final_project.bronze_layer.targets")

# 2. Schema Enforcement / Transformation
df_target_raw = (bronze_df_targets
    .select(
        col("EmployeeID").cast(IntegerType()),
        col("Target").cast(StringType()),
        col("TargetMonth").cast(StringType())
    )
)

In [0]:
df_target_silver = (df_target_raw
    # Clean Currency: Remove '$' and ',' symbols, then cast to Double
    .withColumn("Target", F.regexp_replace(F.col("Target"), r'[\$,]', '').cast(DoubleType()))

    # Clean Date (e.g., 'Friday, December 1, 2017' -> 2017-12-01)
    # Extract only the "Month Day, Year" part to avoid parsing issues
    .withColumn("CleanDate", F.substring_index(F.col("TargetMonth"), ", ", -2))
    .withColumn("TargetMonth", F.to_date(F.col("CleanDate"), "MMMM d, yyyy"))
    .drop("CleanDate") # Drop temporary column
    .dropDuplicates()
    .drop("_rescued_data")
)

In [0]:
display(df_target_silver)

In [0]:
# Check the data types of all columns in the DataFrame
df_target_silver.printSchema()

# Display the first 10 rows to visually inspect the values (e.g., date formats, currency symbols)
df_target_silver.show(10, truncate=False)

In [0]:
# --- OUTLIER & ANOMALY HANDLING ---
# Keep only valid records: Target must be positive (no negative targets) and not null
df_target_clean = df_target_silver.filter(
    (F.col("Target").isNotNull()) & 
    (F.col("Target") > 0) &
    (F.col("TargetMonth").isNotNull())
)

## **Transform to DELTA**

In [0]:
# Targets DELTA
df_target_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("accenture_final_project.silver_layer.targets")